#DEMANDAS TI

### CONFIGURAÇÃO E CARREGAMENTO DE DATASET

In [17]:
# ============================================================
# Importa as bibliotecas do Python
# ============================================================

# Importar o pandas
import pandas as pd

# Importar o LabelEncoder da biblioteca scikit-learn
from sklearn.preprocessing import LabelEncoder

In [18]:
# ============================================================
# Carregar o dataset a partir do arquivo Excel
# ============================================================

# url do arquivo no GitHub
url_dataset = "https://raw.githubusercontent.com/gilbertoag2007/machine-learning-demandas-ti/main/DEMANDAS_DOWNSTREAM_TESTE.xlsx"

# Cria um dataframe com o conteúdo do dataset
df_original = pd.read_excel(url_dataset)

# Lista as 5 primeiras colunas do dataframe.

df_original.shape
df_original.head()

,ID_DEMANDA,SISTEMA,TIPO_DEMANDA,DATA_INICIO_PREVISTA,DATA_INICIO_REALIZADA,DATA_FIM_PREVISTA,DATA_FIM_REALIZADA,STATUS_FINAL
0,1,DFE,BUG_IMPEDITIVO,22/01/2026,22/01/2026,2026-01-24,2026-01-24,DENTRO DO PRAZO
1,2,DFE,BUG_IMPEDITIVO,2026-01-05 00:00:00,2026-01-07 00:00:00,2026-01-07,2026-01-07,DENTRO DO PRAZO
2,3,DFE,BUG_IMPEDITIVO,23/02/2026,23/02/2026,2026-02-25,2026-02-25,DENTRO DO PRAZO
3,4,DFE,BUG_IMPEDITIVO,17/02/2026,17/02/2026,2026-02-19,2026-02-19,DENTRO DO PRAZO
4,5,DFE,BUG_IMPEDITIVO,13/02/2026,13/02/2026,2026-02-15,2026-02-15,DENTRO DO PRAZO


##AJUSTES INICIAIS NO DATAFRAME

In [19]:
# ============================================================
# TÉCNICA: Label Encoding (Codificação de Rótulos)
# ============================================================

# Cria uma cópia do dataframe original para preservá-lo intacto
# Todas as alterações serão feitas apenas no df_ajustado
df_ajustado = df_original.copy()

# Cria uma nova coluna numérica baseada na coluna STATUS_FINAL
# map() substitui cada valor categórico pelo número correspondente
df_ajustado['STATUS_FINAL_NUM'] = df_ajustado['STATUS_FINAL'].map({
    'ATRASO'         : 1,
    'DENTRO DO PRAZO': 0
})

# Variavel Target
target = "STATUS_FINAL_NUM"



In [20]:
# ============================================================
# TÉCNICA: One-Hot Encoding
# ============================================================

# Aplicar One-Hot Encoding na coluna SISTEMA
# pd.get_dummies() cria uma coluna binária (0 ou 1) para cada sistema único
# dtype=int garante que os valores sejam inteiros ao invés de booleanos
# Aplicar nas colunas categóricas sem ordem natural
for coluna in ['SISTEMA', 'TIPO_DEMANDA']:
    dummies = pd.get_dummies(df_ajustado[coluna], prefix=coluna, dtype=int)
    df_ajustado = pd.concat([df_ajustado, dummies], axis=1)
    df_ajustado = df_ajustado.drop(columns=[coluna])


In [21]:

# ============================================================
# CONVERTER COLUNAS DE DATA PARA DATETIME
# Necessário para realizar operações matemáticas entre datas
# ============================================================
colunas_data = [
    'DATA_INICIO_PREVISTA',
    'DATA_INICIO_REALIZADA',
    'DATA_FIM_PREVISTA',
    'DATA_FIM_REALIZADA'
]

for coluna in colunas_data:
    df_ajustado[coluna] = pd.to_datetime(
        df_ajustado[coluna], dayfirst=True, errors='coerce'
    )


In [22]:
# ============================================================
# GERAR DATA_REFERENCIA
# Ponto médio entre início e fim — simula o momento de acompanhamento de cada demanda
# ============================================================

# Calcular ponto médio como referência padrão
# usando DATA_INICIO_PREVISTA e DATA_FIM_PREVISTA
df_ajustado['DATA_REFERENCIA'] = (
    df_ajustado['DATA_INICIO_PREVISTA'] +
    (df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_INICIO_PREVISTA']) / 2
)

# Ajustar referência para demandas já iniciadas
# usar ponto médio entre DATA_INICIO_REALIZADA e DATA_FIM_PREVISTA
mask_iniciadas = df_ajustado['DATA_INICIO_REALIZADA'].notna()

df_ajustado.loc[mask_iniciadas, 'DATA_REFERENCIA'] = (
    df_ajustado.loc[mask_iniciadas, 'DATA_INICIO_REALIZADA'] +
    (
        df_ajustado.loc[mask_iniciadas, 'DATA_FIM_PREVISTA'] -
        df_ajustado.loc[mask_iniciadas, 'DATA_INICIO_REALIZADA']
    ) / 2
)

In [23]:
# ============================================================
# GERAR FEATURES NUMÉRICAS
# ============================================================

# ----------------------------------------------------------
# Duração total planejada da demanda em dias
# Indica o tamanho e complexidade da demanda
# ----------------------------------------------------------
df_ajustado['DURACAO_PREVISTA_DIAS'] = (
    df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days

# ----------------------------------------------------------
# Quantidade de dias de atraso no início da demanda
# Valores positivos indicam atraso no início
# Preenchido com 0 quando não há DATA_INICIO_REALIZADA
# ----------------------------------------------------------
df_ajustado['ATRASO_INICIO_DIAS'] = (
    df_ajustado['DATA_INICIO_REALIZADA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days.fillna(0)

# ----------------------------------------------------------
# Dias restantes até o prazo final na data de referência
# Valores negativos indicam que o prazo já foi ultrapassado
# ----------------------------------------------------------
df_ajustado['DIAS_RESTANTES'] = (
    df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_REFERENCIA']
).dt.days

# ----------------------------------------------------------
# Percentual do prazo consumido até a data de referência
# Indica o quanto do tempo planejado já foi utilizado
# ----------------------------------------------------------
df_ajustado['PERC_TEMPO_DECORRIDO'] = (
    (df_ajustado['DATA_REFERENCIA'] - df_ajustado['DATA_INICIO_PREVISTA']).dt.days /
     df_ajustado['DURACAO_PREVISTA_DIAS']
) * 100

# ----------------------------------------------------------
# Dias sem iniciar após a DATA_INICIO_PREVISTA
# clip(lower=0) evita valores negativos para demandas
# que ainda não atingiram a data de início prevista
# ----------------------------------------------------------
df_ajustado['DIAS_SEM_INICIAR'] = (
    df_ajustado['DATA_REFERENCIA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days.clip(lower=0)


In [24]:
# ============================================================
# GERAR FLAGS BINÁRIAS (0 ou 1)
# ============================================================

# ----------------------------------------------------------
# Flag: a demanda já foi iniciada?
# 1 = sim | 0 = não
# ----------------------------------------------------------
df_ajustado['FLAG_INICIADA'] = (
    df_ajustado['DATA_INICIO_REALIZADA'].notna()
).astype(int)

# ----------------------------------------------------------
# Flag: a demanda atrasou no início?
# 1 = começou depois do previsto | 0 = não
# ----------------------------------------------------------
df_ajustado['FLAG_ATRASO_INICIO'] = (
    df_ajustado['ATRASO_INICIO_DIAS'] > 0
).astype(int)

# ----------------------------------------------------------
# Flag: a demanda deveria ter iniciado mas ainda não iniciou?
# 1 = passou da DATA_INICIO_PREVISTA sem início registrado
# 0 = ainda dentro do prazo de início ou já iniciada
# ----------------------------------------------------------
df_ajustado['FLAG_NAO_INICIADA_NO_PRAZO'] = (
    (df_ajustado['DATA_REFERENCIA'] >= df_ajustado['DATA_INICIO_PREVISTA']) &
    (df_ajustado['DATA_INICIO_REALIZADA'].isna())
).astype(int)

In [26]:
# ============================================================
# REMOVER COLUNAS QUE NÃO DEVEM ENTRAR NO MODELO ANTES DO TREINAMENTO
# ============================================================

colunas_remover = [
    'ID_DEMANDA',
    'STATUS_FINAL',
    'DATA_INICIO_PREVISTA',
    'DATA_INICIO_REALIZADA',
    'DATA_FIM_PREVISTA',
    'DATA_FIM_REALIZADA',
    'DATA_REFERENCIA'
]
df_ajustado = df_ajustado.drop(columns=colunas_remover)

In [28]:
# Exibir resumo das colunas geradas e seus tipos
print('📊 Colunas do dataframe ajustado:')
print(df_ajustado.dtypes)
print(f'\n✅ Shape final: {df_ajustado.shape[0]} linhas x {df_ajustado.shape[1]} colunas')

df_ajustado.head(50)

📊 Colunas do dataframe ajustado:
STATUS_FINAL_NUM                     int64
SISTEMA_CSA                          int64
SISTEMA_DFE                          int64
SISTEMA_DPP                          int64
SISTEMA_SIGAF                        int64
SISTEMA_SIMP                         int64
TIPO_DEMANDA_BUG_IMPEDITIVO          int64
TIPO_DEMANDA_BUG_NAO_IMPEDITIVO      int64
TIPO_DEMANDA_MELHORIA_MEDIA          int64
TIPO_DEMANDA_MELHORIA_PEQUENA        int64
TIPO_DEMANDA_ORIENTACAO              int64
DURACAO_PREVISTA_DIAS                int64
ATRASO_INICIO_DIAS                   int64
DIAS_RESTANTES                       int64
PERC_TEMPO_DECORRIDO               float64
DIAS_SEM_INICIAR                     int64
FLAG_INICIADA                        int64
FLAG_ATRASO_INICIO                   int64
FLAG_NAO_INICIADA_NO_PRAZO           int64
dtype: object

✅ Shape final: 1500 linhas x 19 colunas


,STATUS_FINAL_NUM,SISTEMA_CSA,SISTEMA_DFE,SISTEMA_DPP,SISTEMA_SIGAF,SISTEMA_SIMP,TIPO_DEMANDA_BUG_IMPEDITIVO,TIPO_DEMANDA_BUG_NAO_IMPEDITIVO,TIPO_DEMANDA_MELHORIA_MEDIA,TIPO_DEMANDA_MELHORIA_PEQUENA,TIPO_DEMANDA_ORIENTACAO,DURACAO_PREVISTA_DIAS,ATRASO_INICIO_DIAS,DIAS_RESTANTES,PERC_TEMPO_DECORRIDO,DIAS_SEM_INICIAR,FLAG_INICIADA,FLAG_ATRASO_INICIO,FLAG_NAO_INICIADA_NO_PRAZO
0,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
1,0,0,1,0,0,0,1,0,0,0,0,2,2,0,100.0,2,1,1,0
2,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
3,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
4,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
5,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
6,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
7,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
8,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
9,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
